# RayOps Getting Started Guide

This notebook walks through a complete Ray cluster lifecycle using the RayOps SDK: configure credentials, inspect the SDK, verify connectivity, create a Ray cluster, wait for it to become ready, review its details and endpoints, and clean up.

## Why This Matters
- The focus is **RayOps**: a single SDK provisions, inspects, and tears down Ray clusters — the distributed compute behind model training, hyperparameter tuning, and batch inference.
- A **single JWT bearer token authenticates every request** — you provide it securely when prompted.
- Enter your connection details once (you are prompted for only `BASE_URL`, the token, and an `SSL_VERIFY` choice), then reuse them across every section.
- Every cluster is built from a typed `RayClusterCreate` model, so sizing and node-group selection are explicit, validated, and easy to re-run.

## What You Will Accomplish
**Across one Ray cluster lifecycle (Sections 2–8):**
- Inspect the SDK surface with **`blueprint()`** and confirm connectivity with a **node-group health check**.
- Create a **Ray cluster** from a typed `RayClusterCreate` model and wait for it to become **ready**.
- Review the cluster's **details** and **connection endpoints** (the Ray client and dashboard URLs).
- **List** clusters to confirm the new one, then **delete** it to clean up.

---

## Table of Contents

### [1. Setup & Prerequisites](#1-setup--prerequisites)
- 1.1 Import required modules
- 1.2 Authentication
- 1.3 Configure credentials
- 1.4 Demo configuration *(shared settings)*
- 1.5 Initialize the SDK client

### [2. Inspect Blueprint](#2-inspect-blueprint)

### [3. Health Check](#3-health-check)

### [4. Review Existing Clusters](#4-review-existing-clusters)

### [5. Create a Ray Cluster](#5-create-a-ray-cluster)
- 5.1 Build the cluster request
- 5.2 Submit the request

### [6. Wait Until the Cluster Is Ready](#6-wait-until-the-cluster-is-ready)

### [7. Retrieve Cluster Details](#7-retrieve-cluster-details)
- 7.1 Get cluster details and endpoints
- 7.2 List all clusters

### [8. Clean Up](#8-clean-up)
- 8.1 Delete the cluster
- 8.2 Verify removal

### [9. Reference](#9-reference)
- 9.1 Data models
- 9.2 Built-in help

<a id="setup"></a>
## 1. Setup & Prerequisites

### 1.1 Import Required Modules

Imports the public RayOps SDK classes and the `RayClusterCreate` request model, and defines a small `show_output()` helper so responses are easy to read.


In [44]:
from getpass import getpass
from pprint import pprint

from teradata_agentstack import BearerAuth
from teradata_agentstack.rayops import (
    blueprint,
    RayClusterManagementClient,
    RayClusters,
    NodeGroups,
)
from teradata_agentstack.rayops.models import RayClusterCreate


def show_output(label, value):
    """Pretty-print an SDK response (Pydantic model or plain value)."""
    print(f"\n{label}:")
    formatter = getattr(value, "model_dump", None) or getattr(value, "dict", None) or (lambda: value)
    pprint(formatter() if callable(formatter) else formatter, sort_dicts=False, width=120)


print("Imports loaded.")

Imports loaded.


### 1.2 Authentication

The `teradata_agentstack` RayOps client supports **4 authentication modes** and **3 ways to provide credentials**.

The credential sources are resolved in this order: direct `auth_data` parameter, environment variables, then YAML config file.

#### Authentication Modes

| Auth Mode | Class | Required Fields |
| --- | --- | --- |
| Bearer Token | `BearerAuth` | `auth_bearer` |
| Basic Auth | `BasicAuth` | `username`, `password` |
| Client Credentials (OAuth2) | `ClientCredentialsAuth` | `auth_token_url`, `auth_client_id`, `auth_client_secret` |
| Device Code (OAuth2) | `DeviceCodeAuth` | `auth_token_url`, `auth_client_id`, `auth_client_secret`, `auth_device_auth_url` |

This notebook uses `BearerAuth` for RayOps requests. You enter the token securely via `getpass` in step 1.3 — it is never hardcoded.

### 1.3 Configure Credentials

Prompts for your `BASE_URL` and your auth token (entered securely via `getpass`). Finally, choose whether to verify the endpoint's TLS certificate (`SSL_VERIFY`) — press Enter to keep the default of `false`.

In [47]:
# Endpoint for the RayOps service you are targeting.
BASE_URL = getpass("Enter BASE_URL: ").strip()

# Bearer token (JWT) for the RayOps service, entered securely.
AUTH_TOKEN = getpass("Enter AUTH_TOKEN: ").strip()

# Verify the endpoint's TLS certificate? Defaults to False (press Enter to keep).
# Enter "true" only if your endpoint uses a trusted TLS certificate.
SSL_VERIFY = input("Verify SSL certificate? [true/false] (default false): ").strip().lower() in ("true", "yes", "y", "1")

print("Credentials ready.")
print(f"BASE_URL configured:   {bool(BASE_URL)}")
print(f"AUTH_TOKEN configured: {bool(AUTH_TOKEN)}")
print(f"SSL_VERIFY:            {SSL_VERIFY}")

Enter BASE_URL:  ········
Enter AUTH_TOKEN:  ········
Verify SSL certificate? [true/false] (default false):  F


Credentials ready.
BASE_URL configured:   True
AUTH_TOKEN configured: True
SSL_VERIFY:            False


### 1.4 Demo Configuration

Holds the settings the demo uses to create a cluster: the cluster's identity, its size, and how long to wait for it to become ready. Sensible defaults are provided so you can run the notebook as-is — edit any value to customize your cluster. Leave `NODEGROUP` blank to automatically use the first available node group from the health check. If you change the sizing, stay within the SDK minimums: at least 2 workers, head/worker CPU ≥ 2, and head/worker memory ≥ 4 (Gi).

In [48]:
# ── Cluster identity ────────────────────────────────────────────────────────
CLUSTER_NAME = "rayops-getting-started"

# Node group to schedule on. Leave blank to auto-select the first available one.
NODEGROUP = ""

# ── Cluster sizing (SDK minimums: workers >= 2, CPU >= 2, memory >= 4 Gi) ────
NUM_WORKERS = 2
HEAD_CPU = 2
HEAD_MEMORY = 4      # Gi
WORKER_CPU = 2
WORKER_MEMORY = 4    # Gi

# ── How long to wait for the cluster to become ready ────────────────────────
POLL_INTERVAL_SECONDS = 15
POLL_MAX_ATTEMPTS = 20   # 20 x 15s = up to 5 minutes

print(f"Demo configuration loaded. Cluster name: {CLUSTER_NAME}")

Demo configuration loaded. Cluster name: rayops-getting-started


### 1.5 Initialize the SDK Client

Creates the authenticated `RayClusterManagementClient` and the two resource objects used throughout this notebook — `RayClusters` for cluster operations and `NodeGroups` for node-group lookups.

In [49]:
auth = BearerAuth(auth_bearer=AUTH_TOKEN)
client = RayClusterManagementClient(
    base_url=BASE_URL,
    auth_data=auth,
    ssl_verify=SSL_VERIFY,
)

clusters = RayClusters(client=client)
nodegroups = NodeGroups(client=client)

print("RayOps SDK client initialized.")

RayOps SDK client initialized.


<a id="blueprint"></a>
## 2. Inspect Blueprint

Prints a summary of the available RayOps SDK classes and operations. Run this once to see what the SDK offers before you start creating resources. This is a connection-free call.

In [50]:
blueprint()

----------------------------------------------------------------
Available classes for Ray Management Service SDK:
    * teradata_agentstack.rayops.RayClusters
    * teradata_agentstack.rayops.NodeGroups
----------------------------------------------------------------


<a id="health-check"></a>
## 3. Health Check

Verifies that the RayOps service is reachable and your credentials are accepted by listing the node groups available to your account. A successful response means you are ready to create clusters.

In [51]:
nodegroup_response = nodegroups.list()
show_output("Node Groups", nodegroup_response)


Node Groups:
{'nodegroups': [{'name': 'gpu-optimized',
                 'status': 'ACTIVE',
                 'min_size': 4,
                 'max_size': 4,
                 'labels': {'node.kubernetes.io/role': 'gpu-optimized',
                            'nvidia.com/gpu.family': 'blackwell',
                            'node.kubernetes.io/type': 'td-gpu',
                            'nvidia.com/gpu.count': '3',
                            'node-group': 'gpu-workers',
                            'node-type': 'gpu'},
                 'tags': {'provider': 'onprem'},
                 'taints': [{'key': 'node.kubernetes.io/type', 'value': 'td-gpu', 'effect': 'NoSchedule'}],
                 'created_at': datetime.datetime(2026, 5, 10, 7, 44, 35, tzinfo=TzInfo(0))},
                {'name': 'workload-tier',
                 'status': 'ACTIVE',
                 'min_size': 2,
                 'max_size': 2,
                 'labels': {'node.kubernetes.io/role': 'workload-tier',
            

Formats the results into a clean table showing each node group's name, current status, and minimum and maximum worker count.

In [52]:
if hasattr(nodegroup_response, "nodegroups") and nodegroup_response.nodegroups:
    print(f"{'Name':<30} {'Status':<12} {'Min':>4} {'Max':>4}")
    print("-" * 55)
    for item in nodegroup_response.nodegroups:
        print(f"{item.name:<30} {item.status:<12} {item.min_size:>4} {item.max_size:>4}")
else:
    print("No node groups were returned.")

Name                           Status        Min  Max
-------------------------------------------------------
gpu-optimized                  ACTIVE          4    4
workload-tier                  ACTIVE          2    2


<a id="clusters"></a>
## 4. Review Existing Clusters

Lists the clusters that already exist in your environment. This gives you a clear before/after comparison once you create a new one.

In [53]:
existing_clusters = clusters.list()
show_output("Existing Clusters", existing_clusters)


Existing Clusters:
{'clusters': [{'id': '73cdca1d-1080-48f2-b601-5d496ebbe115',
               'name': 'gpu-cluster',
               'num_workers': 2,
               'status': 'ready',
               'ray_version': '2.55.1',
               'head_cpu': 6,
               'head_memory': 10,
               'head_gpu': 1,
               'worker_cpu': 6,
               'worker_memory': 12,
               'worker_gpu': 1,
               'nodegroup': 'gpu-optimized',
               'description': None,
               'created_by': '<user>@teradata.com',
               'ray_client_endpoint': 'ray://<ray-client-endpoint>',
               'ray_dashboard_endpoint': 'https://<dashboard-host>/one-td/dashboard/<cluster-name>/',
               'created_at': datetime.datetime(2026, 6, 23, 1, 12, 20, tzinfo=TzInfo(0))}],
 'total': 1}


<a id="create"></a>
## 5. Create a Ray Cluster

A Ray cluster is provisioned from a typed `RayClusterCreate` request built from the **Demo Configuration** values.

### 5.1 Build the Cluster Request

Builds the `RayClusterCreate` request from the demo settings. If you left `NODEGROUP` blank, the first available node group from the health check is selected automatically.

In [54]:
# Resolve the node group: use NODEGROUP if set, otherwise the first available one
# from the health check. A node group is required to create a cluster.
nodegroup_name = NODEGROUP
if not nodegroup_name and getattr(nodegroup_response, "nodegroups", None):
    nodegroup_name = nodegroup_response.nodegroups[0].name

if not nodegroup_name:
    raise ValueError(
        "No node group is available to schedule the cluster. Set NODEGROUP in the "
        "Demo Configuration cell, or re-run the Health Check (Section 3) to discover one."
    )

cluster_request = RayClusterCreate(
    name=CLUSTER_NAME,
    num_workers=NUM_WORKERS,
    head_cpu=HEAD_CPU,
    head_memory=HEAD_MEMORY,
    worker_cpu=WORKER_CPU,
    worker_memory=WORKER_MEMORY,
    nodegroup=nodegroup_name,
)

print("Cluster request prepared:")
print(f"  Name       : {cluster_request.name}")
print(f"  Node group : {cluster_request.nodegroup}")
print(f"  Workers    : {cluster_request.num_workers}")

Cluster request prepared:
  Name       : rayops-getting-started
  Node group : gpu-optimized
  Workers    : 2


### 5.2 Submit the Request

Submits the request to the service. The response confirms the cluster has been accepted and provisioning has started.

In [55]:
created_cluster = clusters.create(body=cluster_request)
show_output("Created Cluster", created_cluster)

print(f"\nCluster '{getattr(created_cluster, 'name', CLUSTER_NAME)}' creation requested.")


Created Cluster:
{'id': 'be808cdc-dcaa-46f4-a22b-8f80563c6226',
 'name': 'rayops-getting-started',
 'num_workers': 2,
 'status': 'creating',
 'ray_version': '2.55.1',
 'head_cpu': 2,
 'head_memory': 4,
 'head_gpu': 0,
 'worker_cpu': 2,
 'worker_memory': 4,
 'worker_gpu': 0,
 'nodegroup': 'gpu-optimized',
 'description': None,
 'created_by': None,
 'ray_client_endpoint': None,
 'ray_dashboard_endpoint': None,
 'created_at': datetime.datetime(2026, 6, 25, 6, 59, 54, 847412, tzinfo=TzInfo(0))}

Cluster 'rayops-getting-started' creation requested.


<a id="poll"></a>
## 6. Wait Until the Cluster Is Ready

Polls the cluster until it reaches a terminal state. The SDK checks the status repeatedly and returns once the cluster is ready (or running), fails, or the wait limit is reached. Polling uses the interval and attempt limit from the **Demo Configuration** cell (every 15 seconds, up to 20 attempts — about 5 minutes). The terminal states are `ready`, `running`, `failed`, `error`, and `deleted`.

In [56]:
cluster_detail = clusters.poll(
    name=CLUSTER_NAME,
    interval=POLL_INTERVAL_SECONDS,
    max_attempts=POLL_MAX_ATTEMPTS,
)

show_output("Polled Cluster Detail", cluster_detail)

Polling RayClusters 'rayops-getting-started' (every 15s, max 20 attempts)...
  [1/20] Status: creating
  [2/20] Status: ready
  Done - status: ready

Polled Cluster Detail:
{'id': 'be808cdc-dcaa-46f4-a22b-8f80563c6226',
 'name': 'rayops-getting-started',
 'num_workers': 2,
 'status': 'ready',
 'ray_version': '2.55.1',
 'head_cpu': 2,
 'head_memory': 4,
 'head_gpu': None,
 'worker_cpu': 2,
 'worker_memory': 4,
 'worker_gpu': None,
 'nodegroup': 'gpu-optimized',
 'description': None,
 'created_by': '<user>@teradata.com',
 'ray_client_endpoint': 'ray://<ray-client-endpoint>',
 'ray_dashboard_endpoint': 'https://<dashboard-host>/one-td/dashboard/<cluster-name>/',
 'created_at': datetime.datetime(2026, 6, 25, 6, 59, 24, tzinfo=TzInfo(0))}


<a id="details"></a>
## 7. Retrieve Cluster Details

### 7.1 Get Cluster Details and Endpoints

Fetches the full details of your newly created cluster, including its current status and the Ray client and dashboard endpoints (available once the cluster is ready).

In [57]:
cluster_detail = clusters.get(name=CLUSTER_NAME)
show_output("Cluster Detail", cluster_detail)

# Surface the connection endpoints (the URLs you use to reach the running cluster).
client_endpoint = getattr(cluster_detail, "ray_client_endpoint", None)
dashboard_endpoint = getattr(cluster_detail, "ray_dashboard_endpoint", None)
print("\nEndpoints:")
print(f"  Ray client    : {client_endpoint or 'not available yet'}")
print(f"  Ray dashboard : {dashboard_endpoint or 'not available yet'}")


Cluster Detail:
{'id': 'be808cdc-dcaa-46f4-a22b-8f80563c6226',
 'name': 'rayops-getting-started',
 'num_workers': 2,
 'status': 'ready',
 'ray_version': '2.55.1',
 'head_cpu': 2,
 'head_memory': 4,
 'head_gpu': None,
 'worker_cpu': 2,
 'worker_memory': 4,
 'worker_gpu': None,
 'nodegroup': 'gpu-optimized',
 'description': None,
 'created_by': '<user>@teradata.com',
 'ray_client_endpoint': 'ray://<ray-client-endpoint>',
 'ray_dashboard_endpoint': 'https://<dashboard-host>/one-td/dashboard/<cluster-name>/',
 'created_at': datetime.datetime(2026, 6, 25, 6, 59, 24, tzinfo=TzInfo(0))}

Endpoints:
  Ray client    : ray://<ray-client-endpoint>
  Ray dashboard : https://<dashboard-host>/one-td/dashboard/<cluster-name>/


### 7.2 List All Clusters

Lists all clusters again to confirm the new cluster is present. Compare this with the snapshot from Section 4.

In [58]:
updated_clusters = clusters.list()
show_output("All Clusters After Create", updated_clusters)


All Clusters After Create:
{'clusters': [{'id': '73cdca1d-1080-48f2-b601-5d496ebbe115',
               'name': 'gpu-cluster',
               'num_workers': 2,
               'status': 'ready',
               'ray_version': '2.55.1',
               'head_cpu': 6,
               'head_memory': 10,
               'head_gpu': 1,
               'worker_cpu': 6,
               'worker_memory': 12,
               'worker_gpu': 1,
               'nodegroup': 'gpu-optimized',
               'description': None,
               'created_by': '<user>@teradata.com',
               'ray_client_endpoint': 'ray://<ray-client-endpoint>',
               'ray_dashboard_endpoint': 'https://<dashboard-host>/one-td/dashboard/<cluster-name>/',
               'created_at': datetime.datetime(2026, 6, 23, 1, 12, 20, tzinfo=TzInfo(0))},
              {'id': 'be808cdc-dcaa-46f4-a22b-8f80563c6226',
               'name': 'rayops-getting-started',
               'num_workers': 2,
               'status': 'ready',


<a id="cleanup"></a>
## 8. Clean Up

### 8.1 Delete the Cluster

When you are done, delete the cluster you created. The response confirms the deletion request has been accepted.

In [59]:
delete_response = clusters.delete(name=CLUSTER_NAME)
print(f"Delete requested for '{CLUSTER_NAME}': {delete_response}")

Delete requested for 'rayops-getting-started': 


### 8.2 Verify Removal

Lists the clusters again and confirms the deleted cluster no longer appears in your environment.

In [60]:
final_clusters = clusters.list()

remaining = getattr(final_clusters, "clusters", [])
cluster_still_exists = any(c.name == CLUSTER_NAME for c in remaining)

if cluster_still_exists:
    print(f"Note: '{CLUSTER_NAME}' still appears in the cluster list (deletion may still be in progress).")
else:
    print(f"Confirmed: '{CLUSTER_NAME}' has been removed from the cluster list.")

show_output("Final Cluster List", final_clusters)

Confirmed: 'rayops-getting-started' has been removed from the cluster list.

Final Cluster List:
{'clusters': [{'id': '73cdca1d-1080-48f2-b601-5d496ebbe115',
               'name': 'gpu-cluster',
               'num_workers': 2,
               'status': 'ready',
               'ray_version': '2.55.1',
               'head_cpu': 6,
               'head_memory': 10,
               'head_gpu': 1,
               'worker_cpu': 6,
               'worker_memory': 12,
               'worker_gpu': 1,
               'nodegroup': 'gpu-optimized',
               'description': None,
               'created_by': '<user>@teradata.com',
               'ray_client_endpoint': 'ray://<ray-client-endpoint>',
               'ray_dashboard_endpoint': 'https://<dashboard-host>/one-td/dashboard/<cluster-name>/',
               'created_at': datetime.datetime(2026, 6, 23, 1, 12, 20, tzinfo=TzInfo(0))}],
 'total': 1}


<a id="reference"></a>
## 9. Reference

### 9.1 Data Models

#### Ray Cluster Poll States

`RayClusters.poll()` waits until the cluster reaches one of these terminal states (default: every 15 seconds, up to 20 attempts).

| State | Meaning |
|-------|---------|
| `ready` | Cluster is provisioned and ready to use — **success** |
| `running` | Cluster is running — **success** |
| `failed` | Cluster provisioning failed |
| `error` | An error occurred while provisioning |
| `deleted` | Cluster has been removed |

#### `RayClusterCreate` Request Fields

| Field | Required | Default | Description |
|-------|----------|---------|-------------|
| `name` | Yes | – | Unique cluster name |
| `nodegroup` | Yes | – | Node group to schedule the cluster on |
| `num_workers` | No | `2` | Number of worker nodes (minimum 2) |
| `head_cpu` | No | `2` | CPU cores for the head node (minimum 2) |
| `head_memory` | No | `4` | Memory (Gi) for the head node (minimum 4) |
| `head_gpu` | No | `0` | GPUs for the head node |
| `worker_cpu` | No | `2` | CPU cores per worker (minimum 2) |
| `worker_memory` | No | `4` | Memory (Gi) per worker (minimum 4) |
| `worker_gpu` | No | `0` | GPUs per worker |
| `description` | No | `None` | Optional description (max 128 characters) |

#### `RayClusterResponse` Key Fields

| Field | Description |
|-------|-------------|
| `id` | Unique cluster identifier |
| `name` | Cluster name |
| `status` | Current lifecycle status |
| `num_workers` | Number of worker nodes |
| `ray_version` | Ray version running on the cluster |
| `ray_client_endpoint` | URL for the Ray client connection |
| `ray_dashboard_endpoint` | URL for the Ray dashboard |
| `nodegroup` | Node group the cluster is scheduled on |
| `created_at` | Creation timestamp |

### 9.2 Built-in Help

Each cell below prints the built-in documentation for one of the core objects used in this notebook. Run any of them whenever you want a quick reference for the available methods and fields.

In [61]:
# Client: authentication and connection to the RayOps service
help(RayClusterManagementClient)

Help on class RayClusterManagementClient in module teradata_agentstack.rayops._client:

class RayClusterManagementClient(teradata_agentstack.api_client.Client)
 |  RayClusterManagementClient(base_url=None, auth_data=None, ssl_verify=True, config_file=None)
 |  
 |  Method resolution order:
 |      RayClusterManagementClient
 |      teradata_agentstack.api_client.Client
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __init__(self, base_url=None, auth_data=None, ssl_verify=True, config_file=None)
 |      DESCRIPTION:
 |          Initializes the client object and sets up the configuration for the Ray Management Service client.
 |      
 |      PARAMETERS:
 |          base_url:
 |              Optional Argument.
 |              Specifies the base URL of the API endpoint. All requests are made relative to
 |              this URL. It can be provided directly, or derived from the "BASE_URL" environment
 |              variable or the YAML configuration variable "base_url". I

In [62]:
# Cluster operations: create, list, get, delete, and poll
help(RayClusters)

Help on class RayClusters in module teradata_agentstack.rayops:

class RayClusters(builtins.object)
 |  RayClusters(client)
 |  
 |  APIs without a tag.
 |  
 |  Methods defined here:
 |  
 |  __init__ = _constructor(self, client) from teradata_agentstack._utils
 |      Constructor for the dynamic class.
 |      :param client: The client instance to be used by the class.
 |  
 |  create(*c, **kwargs) from teradata_agentstack._utils._create_dynamic_method.<locals>
 |      DESCRIPTION: 
 |          The function 'create' does the following: 
 |          - Create a new Ray cluster (idempotent operation)
 |      
 |      PARAMETERS:
 |          body (Required):
 |              Specifies body description.
 |              Types: RayClusterCreate, dict
 |      
 |          return_dict (Optional):
 |              Specifies whether to return dict. When set to False, schema class objects are returned. Otherwise, the function returns dict.
 |              If the API in the backend does not return 

In [63]:
# Node group operations: list available node groups
help(NodeGroups)

Help on class NodeGroups in module teradata_agentstack.rayops:

class NodeGroups(builtins.object)
 |  NodeGroups(client)
 |  
 |  APIs without a tag.
 |  
 |  Methods defined here:
 |  
 |  __init__ = _constructor(self, client) from teradata_agentstack._utils
 |      Constructor for the dynamic class.
 |      :param client: The client instance to be used by the class.
 |  
 |  list(*c, **kwargs) from teradata_agentstack._utils._create_dynamic_method.<locals>
 |      DESCRIPTION: 
 |               The function 'list' does the following: 
 |               - List all available node groups from the configured provider.
 |      
 |      Returns simplified information about each node group including:
 |      - Node group name and status
 |      - Scaling configuration (min and max sizes)
 |      - Labels for pod scheduling
 |      - Provider tags
 |      - Creation timestamp
 |      
 |      This information can be used to determine where to deploy Ray clusters.
 |      
 |      Provider beh

In [64]:
# Cluster request model: all configurable fields and their defaults
help(RayClusterCreate)

Help on class RayClusterCreate in module teradata_agentstack.rayops.models:

class RayClusterCreate(_RayBaseModel)
 |  RayClusterCreate(*, name: str, nodegroup: str, num_workers: typing.Annotated[int, Ge(ge=2)] = 2, head_cpu: typing.Annotated[int, Ge(ge=2)] = 2, head_memory: typing.Annotated[int, Ge(ge=4)] = 4, head_gpu: Annotated[Optional[int], Ge(ge=0)] = 0, worker_cpu: typing.Annotated[int, Ge(ge=2)] = 2, worker_memory: typing.Annotated[int, Ge(ge=4)] = 4, worker_gpu: Annotated[Optional[int], Ge(ge=0)] = 0, description: Annotated[Optional[str], MaxLen(max_length=128)] = None, **extra_data: Any) -> None
 |  
 |  Request body for creating a new Ray cluster.
 |  
 |  Method resolution order:
 |      RayClusterCreate
 |      _RayBaseModel
 |      pydantic.main.BaseModel
 |      builtins.object
 |  
 |  Data and other attributes defined here:
 |  
 |  __abstractmethods__ = frozenset()
 |  
 |  __annotations__ = {'description': 'Optional[str]', 'head_cpu': 'int', ...
 |  
 |  __class_vars

---

Last validated: 2026-06-23